In [13]:
%%file app.py

import streamlit as st

st.set_page_config(
    page_title="Dashboard ryzyka kredytowego",
    page_icon="💳",
    layout="wide"
)

def strona_startowa():
    st.title("💳 Dashboard ryzyka kredytowego")

    st.markdown("""
Projekt modelowania ryzyka kredytowego.

Autorzy:
- Marcel Joda
- Aleksander Kopera
- Maciej Rak

Dane: Home Credit Default Risk
""")

pg = st.navigation([
    st.Page(strona_startowa, title="Strona startowa", icon="🏠"),
    st.Page("pages/1_Model_Overview.py", title="Podsumowanie modeli", icon="📊"),
    st.Page("pages/2_Exploratory_Data_Analysis.py", title="Eksploracyjna analiza danych", icon="📈"),
    st.Page("pages/3_Client_Explorer.py", title="Eksplorator klientów", icon="👤"),
    st.Page("pages/4_Credit_Scoring.py", title="Scoring kredytowy", icon="💳"),
])
pg.run()


Overwriting app.py


In [14]:
%%file pages/1_Model_Overview.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    confusion_matrix
)
import plotly.graph_objects as go
import plotly.express as px

# Konfiguracja cache'owania danych do krzywych ROC
@st.cache_data
def get_roc_data():
    all_models = joblib.load('all_models.pkl')
    train = pd.read_csv('data/app_train_processed.csv')
    test = pd.read_csv('data/app_valid_processed.csv')
    
    X_train = train.drop(columns=['TARGET'])
    y_train = train['TARGET']
    X_test = test.drop(columns=['TARGET'])
    y_test = test['TARGET']
    
    roc_data = {
        'train': {},
        'test': {}
    }
    
    for name, mdl in all_models.items():
        # Train ROC
        p_tr = mdl.predict_proba(X_train)[:, 1]
        fpr_tr, tpr_tr, _ = roc_curve(y_train, p_tr)
        auc_tr = roc_auc_score(y_train, p_tr)
        step_tr = max(1, len(fpr_tr) // 1000)
        roc_data['train'][name] = {
            'fpr': fpr_tr[::step_tr].tolist(),
            'tpr': tpr_tr[::step_tr].tolist(),
            'auc': float(auc_tr)
        }
        
        # Test ROC
        p_te = mdl.predict_proba(X_test)[:, 1]
        fpr_te, tpr_te, _ = roc_curve(y_test, p_te)
        auc_te = roc_auc_score(y_test, p_te)
        step_te = max(1, len(fpr_te) // 1000)
        roc_data['test'][name] = {
            'fpr': fpr_te[::step_te].tolist(),
            'tpr': tpr_te[::step_te].tolist(),
            'auc': float(auc_te)
        }
        
    return roc_data

st.title("📊 Podsumowanie i porównanie modeli")

# Wczytanie modelu
model = joblib.load("best_credit_model.pkl")
all_models = joblib.load("all_models.pkl")

# Wczytanie danych
df = pd.read_csv("data/app_valid_processed.csv")
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

# ------------------------------------------------------------------------------
# Sekcja 1: Główny Model i parametry odcięcia
# ------------------------------------------------------------------------------

threshold = st.slider(
    "Próg odcięcia",
    min_value=0.1,
    max_value=0.9,
    value=0.6367,
    step=0.01
)

# Predykcje dla wybranego progu
y_prob = model.predict_proba(X)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

# Liczenie metryk
accuracy = accuracy_score(y, y_pred)
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1 = f1_score(y, y_pred)
auc = roc_auc_score(y, y_prob)

# Podstawowe statystyki zbioru
st.write(f"Liczba obserwacji (zbiór testowy): **{len(df):,}**".replace(",", " "))
st.write(f"Liczba zmiennych wejściowych: **{X.shape[1]}**")

# Wyświetlanie metryk w kolumnach (przetłumaczone na język polski)
col1, col2, col3, col4, col5 = st.columns(5)
with col1:
    st.metric("AUC", f"{auc:.4f}")
with col2:
    st.metric("Dokładność", f"{accuracy:.4f}")
with col3:
    st.metric("Precyzja", f"{precision:.4f}")
with col4:
    st.metric("Czułość", f"{recall:.4f}")
with col5:
    st.metric("F1-Score", f"{f1:.4f}")

st.write("---")

# Rozkład target i Confusion Matrix obok siebie w kolumnach
col_dist, col_cm = st.columns(2)

with col_dist:
    st.subheader("Udział klientów defaultujących")
    target_df = y.value_counts(normalize=True).reset_index()
    target_df.columns = ["TARGET", "Udział"]
    fig_target = px.bar(
        target_df,
        x="TARGET",
        y="Udział",
        text=target_df["Udział"].apply(lambda x: f"{x*100:.1f}%"),
        color_discrete_sequence=["#1f77b4"]
    )
    fig_target.update_layout(
        xaxis_title="Default",
        yaxis_title="Udział",
        xaxis=dict(tickmode="array", tickvals=[0, 1], ticktext=["Brak defaultu (0)", "Default (1)"]),
        height=350,
        margin=dict(l=20, r=20, t=30, b=20)
    )
    st.plotly_chart(fig_target, use_container_width=True)

with col_cm:
    st.subheader("Macierz błędu (Confusion Matrix)")
    cm = confusion_matrix(y, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Brak defaultu (0)", "Default (1)"],
        columns=["Brak defaultu (Pred 0)", "Default (Pred 1)"]
    )
    cm_text = [[f"{val:,}".replace(",", " ") for val in row] for row in cm]
    
    fig_cm = px.imshow(
        cm_df,
        aspect="auto",
        color_continuous_scale="Blues",
        labels=dict(x="Predykcja", y="Rzeczywistość", color="Liczba")
    )
    fig_cm.update_traces(text=cm_text, texttemplate="%{text}", textfont=dict(size=14))
    fig_cm.update_layout(
        height=350,
        margin=dict(l=20, r=20, t=30, b=20),
        coloraxis_showscale=False
    )
    st.plotly_chart(fig_cm, use_container_width=True)

# ------------------------------------------------------------------------------
# Sekcja 2: Porównanie Modeli
# ------------------------------------------------------------------------------
st.write("---")
st.subheader("📊 Porównanie wyników modeli (Trening vs Test)")

# Wczytanie prekalkulowanych metryk
df_metrics = pd.read_csv("data/model_comparison_results.csv")

# Tłumaczenie nazw kolumn i filtrowanie (usunięto zysk finansowy i optymalny próg)
df_metrics_pl = df_metrics.rename(columns={
    "Model": "Model",
    "Dataset": "Zbiór danych",
    "AUC": "AUC",
    "Accuracy": "Dokładność",
    "Precision": "Precyzja",
    "Recall": "Czułość",
    "F1": "F1-Score"
})

# Zachowanie tylko wymaganych kolumn
df_metrics_pl = df_metrics_pl[["Model", "Zbiór danych", "AUC", "Dokładność", "Precyzja", "Czułość", "F1-Score"]]

st.dataframe(
    df_metrics_pl.style.format({
        "AUC": "{:.4f}",
        "Dokładność": "{:.4f}",
        "Precyzja": "{:.4f}",
        "Czułość": "{:.4f}",
        "F1-Score": "{:.4f}"
    }),
    use_container_width=True,
    hide_index=True
)

# Porównanie krzywych ROC (Trening vs Test)
st.write("### Krzywe ROC (Trening vs Test)")
try:
    roc_data = get_roc_data()
    col_roc_tr, col_roc_te = st.columns(2)
    
    with col_roc_tr:
        st.write("**Zbiór Treningowy**")
        fig_roc_tr = go.Figure()
        fig_roc_tr.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode="lines",
            line=dict(dash="dash", color="gray"),
            showlegend=False
        ))
        for model_name, data in roc_data["train"].items():
            fig_roc_tr.add_trace(go.Scatter(
                x=data["fpr"],
                y=data["tpr"],
                mode="lines",
                name=f"{model_name} (AUC = {data['auc']:.4f})"
            ))
        fig_roc_tr.update_layout(
            xaxis_title="False Positive Rate",
            yaxis_title="True Positive Rate",
            height=350,
            margin=dict(l=20, r=20, t=30, b=20),
            xaxis=dict(range=[0, 1], constrain="domain"),
            yaxis=dict(scaleanchor="x", scaleratio=1, range=[0, 1], constrain="domain"),
        )
        st.plotly_chart(fig_roc_tr, use_container_width=True)
        
    with col_roc_te:
        st.write("**Zbiór Testowy**")
        fig_roc_te = go.Figure()
        fig_roc_te.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode="lines",
            line=dict(dash="dash", color="gray"),
            showlegend=False
        ))
        for model_name, data in roc_data["test"].items():
            fig_roc_te.add_trace(go.Scatter(
                x=data["fpr"],
                y=data["tpr"],
                mode="lines",
                name=f"{model_name} (AUC = {data['auc']:.4f})"
            ))
        fig_roc_te.update_layout(
            xaxis_title="False Positive Rate",
            yaxis_title="True Positive Rate",
            height=350,
            margin=dict(l=20, r=20, t=30, b=20),
            xaxis=dict(range=[0, 1], constrain="domain"),
            yaxis=dict(scaleanchor="x", scaleratio=1, range=[0, 1], constrain="domain"),
        )
        st.plotly_chart(fig_roc_te, use_container_width=True)
        
except Exception as e:
    st.warning(f"Nie udało się załadować danych do krzywych ROC: {e}")


Overwriting pages/1_Model_Overview.py


In [15]:
%%file pages/2_Exploratory_Data_Analysis.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

st.title("📈 Eksploracyjna analiza danych")

# --- Funkcje do cache'owania danych ---
@st.cache_data
def load_processed_data():
    return pd.read_csv("data/app_train_processed.csv")

@st.cache_data
def load_raw_column(col_name):
    # Wczytujemy tylko tę jedną kolumnę w celu optymalizacji pamięci i prędkości
    return pd.read_csv("data/application_train_raw_subset.csv.gz", usecols=[col_name])

# Słownik transformacji przed / po
MAPPING_BEFORE_AFTER = {
    "EXT_SOURCE_1_imp_income_type_gender": {
        "raw_col": "EXT_SOURCE_1",
        "type": "numeric",
        "desc": "Imputacja medianą w podgrupach (Typ dochodu, Płeć) oraz standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1)."
    },
    "EXT_SOURCE_3_imp_education": {
        "raw_col": "EXT_SOURCE_3",
        "type": "numeric",
        "desc": "Imputacja medianą w podgrupach (Wykształcenie) oraz standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1)."
    },
    "DAYS_EMPLOYED_imp_income": {
        "raw_col": "DAYS_EMPLOYED",
        "type": "numeric",
        "desc": "Zastąpienie wartości anomalnych (365243 dni, oznaczających bezrobocie lub specyficzny status emerytalny) wartościami brakującymi, imputacja medianą w grupach typu dochodu oraz standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1)."
    },
    "CODE_GENDER_imputed": {
        "raw_col": "CODE_GENDER",
        "type": "categorical",
        "desc": "Imputacja rzadkich wartości anomalnych (np. 'XNA' zastąpione przez najczęściej występującą płeć: 'F')."
    },
    "NAME_TYPE_SUITE_grouped": {
        "raw_col": "NAME_TYPE_SUITE",
        "type": "categorical",
        "desc": "Połączenie rzadkich kategorii (udział < 5%) w jedną kategorię zbiorczą 'other'."
    },
    "NAME_HOUSING_TYPE_grouped": {
        "raw_col": "NAME_HOUSING_TYPE",
        "type": "categorical",
        "desc": "Połączenie rzadkich kategorii (udział < 5%) w jedną kategorię zbiorczą 'other'."
    },
    "OCCUPATION_TYPE_grouped": {
        "raw_col": "OCCUPATION_TYPE",
        "type": "categorical",
        "desc": "Połączenie kategorii 'Core staff' oraz braków danych w nową wspólną kategorię 'missing / core staff'."
    },
    "ORGANIZATION_TYPE_grouped": {
        "raw_col": "ORGANIZATION_TYPE",
        "type": "categorical",
        "desc": "Połączenie rzadkich kategorii (udział < 5%) w jedną kategorię zbiorczą 'other'."
    },
    "NAME_INCOME_TYPE_grouped": {
        "raw_col": "NAME_INCOME_TYPE",
        "type": "categorical",
        "desc": "Grupowanie rzadkich kategorii oraz 'State servant' w jedną kategorię 'State_servant'."
    },
    "NAME_FAMILY_STATUS_grouped": {
        "raw_col": "NAME_FAMILY_STATUS",
        "type": "categorical",
        "desc": "Połączenie kategorii: 'Civil marriage' oraz 'Single / not married' w grupę 'Civil marriage / Single / not married', a 'Unknown' oraz 'Married' w grupę 'Married'."
    },
    "CNT_CHILDREN_grouped": {
        "raw_col": "CNT_CHILDREN",
        "type": "categorical",
        "desc": "Grupowanie liczby dzieci: wartości 0 i 1 pozostawiono bez zmian, natomiast wszystkie wartości >= 2 połączono w kategorię '>=2'."
    },
    "OBS_30_CNT_SOCIAL_CIRCLE_grouped": {
        "raw_col": "OBS_30_CNT_SOCIAL_CIRCLE",
        "type": "categorical",
        "desc": "Grupowanie liczby obserwacji w otoczeniu klienta (30 dni): wartości 0 i 1 pozostawiono bez zmian, natomiast wszystkie wartości >= 2 połączono w kategorię '>=2'."
    },
    "OBS_60_CNT_SOCIAL_CIRCLE_grouped": {
        "raw_col": "OBS_60_CNT_SOCIAL_CIRCLE",
        "type": "categorical",
        "desc": "Grupowanie liczby obserwacji w otoczeniu klienta (60 dni): wartości 0 i 1 pozostawiono bez zmian, natomiast wszystkie wartości >= 2 połączono w kategorię '>=2'."
    },
    "AMT_REQ_CREDIT_BUREAU_YEAR_grouped": {
        "raw_col": "AMT_REQ_CREDIT_BUREAU_YEAR",
        "type": "categorical",
        "desc": "Grupowanie liczby zapytań do BIK (rok): wartości 0 i 1 pozostawiono bez zmian, 2 i 3 połączono w '2-3', wartości >= 4 w '>=4', a braki danych oznaczono jako 'missing'."
    },
    "EXT_SOURCE_2": {
        "raw_col": "EXT_SOURCE_2",
        "type": "numeric",
        "desc": "Uzupełnienie braków danych (imputacja ogólną medianą) oraz standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1)."
    },
    "AMT_ANNUITY": {
        "raw_col": "AMT_ANNUITY",
        "type": "numeric",
        "desc": "Uzupełnienie braków danych (imputacja ogólną medianą) oraz standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1)."
    },
    "DAYS_LAST_PHONE_CHANGE": {
        "raw_col": "DAYS_LAST_PHONE_CHANGE",
        "type": "numeric",
        "desc": "Uzupełnienie braków danych (imputacja ogólną medianą) oraz standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1)."
    },
    "edu_higher_academic": {
        "raw_col": "NAME_EDUCATION_TYPE",
        "type": "categorical",
        "desc": "Zmienna binarna: 1 dla wykształcenia 'Higher education' lub 'Academic degree', 0 w pozostałych przypadkach."
    },
    "2_fam_members": {
        "raw_col": "CNT_FAM_MEMBERS",
        "type": "categorical",
        "desc": "Zmienna binarna: 1 jeśli liczba członków rodziny wynosi dokładnie 2, 0 w pozostałych przypadkach."
    },
    "DEF_60_CNT_SOCIAL_CIRCLE_eq_0": {
        "raw_col": "DEF_60_CNT_SOCIAL_CIRCLE",
        "type": "categorical",
        "desc": "Zmienna binarna: 1 jeśli liczba spóźnień w otoczeniu (60 dni) wynosi 0, 0 w pozostałych przypadkach."
    },
    "AMT_CREDIT": {
        "raw_col": "AMT_CREDIT",
        "type": "numeric",
        "desc": "Standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1) surowej kwoty kredytu."
    },
    "AMT_INCOME_TOTAL": {
        "raw_col": "AMT_INCOME_TOTAL",
        "type": "numeric",
        "desc": "Standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1) surowego całkowitego dochodu."
    },
    "DAYS_BIRTH": {
        "raw_col": "DAYS_BIRTH",
        "type": "numeric",
        "desc": "Brak zmian (wiek w dniach przeniesiony bezpośrednio z danych surowych)."
    },
    "DAYS_REGISTRATION": {
        "raw_col": "DAYS_REGISTRATION",
        "type": "numeric",
        "desc": "Standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1) czasu od rejestracji."
    },
    "DAYS_ID_PUBLISH": {
        "raw_col": "DAYS_ID_PUBLISH",
        "type": "numeric",
        "desc": "Brak zmian (czas od wydania dowodu przeniesiony bezpośrednio z danych surowych)."
    },
    "REGION_POPULATION_RELATIVE": {
        "raw_col": "REGION_POPULATION_RELATIVE",
        "type": "numeric",
        "desc": "Standaryzacja Z-score (normalizacja do średniej 0 i odchylenia std 1) względnej populacji regionu."
    },
    "NAME_CONTRACT_TYPE": {
        "raw_col": "NAME_CONTRACT_TYPE",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "FLAG_OWN_CAR": {
        "raw_col": "FLAG_OWN_CAR",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "FLAG_OWN_REALTY": {
        "raw_col": "FLAG_OWN_REALTY",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "FLAG_EMP_PHONE": {
        "raw_col": "FLAG_EMP_PHONE",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "FLAG_WORK_PHONE": {
        "raw_col": "FLAG_WORK_PHONE",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "FLAG_CONT_MOBILE": {
        "raw_col": "FLAG_CONT_MOBILE",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "FLAG_PHONE": {
        "raw_col": "FLAG_PHONE",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "FLAG_EMAIL": {
        "raw_col": "FLAG_EMAIL",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "REGION_RATING_CLIENT": {
        "raw_col": "REGION_RATING_CLIENT",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "REG_REGION_NOT_LIVE_REGION": {
        "raw_col": "REG_REGION_NOT_LIVE_REGION",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "REG_REGION_NOT_WORK_REGION": {
        "raw_col": "REG_REGION_NOT_WORK_REGION",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "REG_CITY_NOT_LIVE_CITY": {
        "raw_col": "REG_CITY_NOT_LIVE_CITY",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    },
    "LIVE_CITY_NOT_WORK_CITY": {
        "raw_col": "LIVE_CITY_NOT_WORK_CITY",
        "type": "categorical",
        "desc": "Brak zmian (zmienna przeniesiona bezpośrednio z danych surowych)."
    }
}

# Wczytanie przetworzonych danych
df = load_processed_data()

variables = [c for c in df.columns if c != "TARGET"]

selected = st.selectbox(
    "Wybierz zmienną",
    variables
)

st.write("Wybrana zmienna:")
st.write(selected)

st.write("Typ zmiennej:")
st.write(str(df[selected].dtype))

# Sprawdzenie typu zmiennej i unikalnych wartości (czy kategoryczna lub binarna 0/1)
unique_vals = df[selected].dropna().unique()
is_categorical_or_binary = (not pd.api.types.is_numeric_dtype(df[selected])) or (len(unique_vals) <= 2)

# Wygładzenie wykresu gęstości (KDE bandwidth adjustment) - stała wartość 2.5
bw_adjust = 2.5

# Funkcja pomocnicza do tworzenia gładkiego KDE z wypełnieniem obszaru pod wykresem
def plot_kde(data_list, name_list, color_list=None, bw_adjust=2.5):
    fig = go.Figure()
    for i, (data, name) in enumerate(zip(data_list, name_list)):
        clean_data = data.dropna()
        if len(clean_data) < 2:
            continue

        # Obliczenie gęstości za pomocą scipy stats
        kde = gaussian_kde(clean_data)
        # Zwiększenie wygładzania
        kde.covariance_factor = lambda: kde.factor * bw_adjust
        kde._compute_covariance()

        # Generowanie punktów dla osi X z marginesem
        x_min, x_max = clean_data.min(), clean_data.max()
        margin = (x_max - x_min) * 0.15 if x_max != x_min else 1.0
        x_grid = np.linspace(x_min - margin, x_max + margin, 300)
        y_grid = kde(x_grid)

        # Wybór koloru i przygotowanie przezroczystego wypełnienia
        color = color_list[i] if color_list else None
        fillcolor = f"rgba({color[4:-1]}, 0.2)" if (color and color.startswith("rgb(")) else None

        fig.add_trace(go.Scatter(
            x=x_grid,
            y=y_grid,
            mode="lines",
            name=name,
            line=dict(width=2.5, color=color),
            fill="tozeroy",
            fillcolor=fillcolor
        ))
    return fig

# --- Wykres 1: Rozkład ogólny ---
st.subheader("Gęstość zmiennych")

if is_categorical_or_binary:
    # Udział procentowy dla kategorycznych / binarnych
    cat_df = df[selected].value_counts(normalize=True).reset_index()
    cat_df.columns = [selected, "Share"]
    cat_df = cat_df.sort_values(by=selected)

    fig = px.bar(
        cat_df, 
        x=selected, 
        y="Share", 
        text=cat_df["Share"].apply(lambda x: f"{x*100:.1f}%")
    )
    fig.update_layout(
        yaxis_title="Gęstość",
        xaxis=dict(type='category') # Wymuszenie kategorii na osi X (zapobiega osiom ułamkowym dla 0/1)
    )
else:
    # Wykres gęstości (KDE)
    fig = plot_kde(
        [df[selected]], 
        [selected], 
        color_list=["rgb(31, 119, 180)"], 
        bw_adjust=bw_adjust
    )
    fig.update_layout(
        xaxis_title=selected, 
        yaxis_title="Gęstość",
        showlegend=False
    )

st.plotly_chart(fig, use_container_width=True)


# --- Wykres 2: Rozkład względem TARGET ---
st.subheader("Funkcja gęstości w podziale na klientów defaultujących i nie-defaultujących")

if is_categorical_or_binary:
    # Wykres słupkowy procentowy z podziałem na TARGET
    cat_target = df.groupby([selected, "TARGET"]).size().unstack(fill_value=0)
    cat_target_pct = cat_target.div(cat_target.sum(axis=0), axis=1).reset_index()

    # Przekształcenie do formatu długiego
    cat_target_melted = cat_target_pct.melt(
        id_vars=selected, 
        value_vars=[0, 1], 
        var_name="TARGET", 
        value_name="Share"
    )
    cat_target_melted["TARGET"] = cat_target_melted["TARGET"].map({0: "Brak defaultu (0)", 1: "Default (1)"})

    fig_target = px.bar(
        cat_target_melted, 
        x=selected, 
        y="Share", 
        color="TARGET", 
        barmode="group",
        text=cat_target_melted["Share"].apply(lambda x: f"{x*100:.1f}%")
    )
    fig_target.update_layout(
        yaxis_title="Share",
        xaxis=dict(type='category')
    )
else:
    # Dwie krzywe gęstości (KDE) na jednym wykresie
    group0 = df[df["TARGET"] == 0][selected]
    group1 = df[df["TARGET"] == 1][selected]

    fig_target = plot_kde(
        [group0, group1], 
        ["Brak defaultu (0)", "Default (1)"], 
        color_list=["rgb(44, 160, 44)", "rgb(214, 39, 40)"], # zielony vs czerwony
        bw_adjust=bw_adjust
    )
    fig_target.update_layout(
        xaxis_title=selected, 
        yaxis_title="Density"
    )

st.plotly_chart(fig_target, use_container_width=True)


# --- Wykres 3: Porównanie przed i po transformacji ---
st.write("---")
st.subheader("🔄 Porównanie rozkładów przed i po transformacji")

selected_transform = st.selectbox(
    "Wybierz zmienną do porównania transformacji",
    options=list(MAPPING_BEFORE_AFTER.keys()),
    format_func=lambda x: f"{x} (oryg. {MAPPING_BEFORE_AFTER[x]['raw_col']})"
)

if selected_transform:
    info = MAPPING_BEFORE_AFTER[selected_transform]
    raw_col = info["raw_col"]
    var_type = info["type"]
    desc = info["desc"]

    st.info(f"**Opis transformacji:** {desc}")

    # Wczytanie surowych danych (tylko potrzebnej kolumny z cache)
    df_raw_col = load_raw_column(raw_col)

    if var_type == "numeric":
        col_left, col_right = st.columns(2)
        raw_data = df_raw_col[raw_col].dropna()
        proc_data = df[selected_transform].dropna()

        # Przed transformacją (surowe)
        with col_left:
            st.write("**Przed transformacją (surowe):**")
            if len(raw_data) >= 2:
                kde_raw = gaussian_kde(raw_data)
                kde_raw.covariance_factor = lambda: kde_raw.factor * bw_adjust
                kde_raw._compute_covariance()
                x_min, x_max = raw_data.min(), raw_data.max()
                margin = (x_max - x_min) * 0.15 if x_max != x_min else 1.0
                x_grid = np.linspace(x_min - margin, x_max + margin, 300)
                y_grid = kde_raw(x_grid)

                fig_left = go.Figure()
                fig_left.add_trace(go.Scatter(
                    x=x_grid,
                    y=y_grid,
                    mode="lines",
                    name="Surowe",
                    line=dict(width=2.5, color="rgb(31, 119, 180)"),
                    fill="tozeroy",
                    fillcolor="rgba(31, 119, 180, 0.2)"
                ))
                fig_left.update_layout(
                    xaxis_title=raw_col,
                    yaxis_title="Gęstość",
                    showlegend=False
                )
                st.plotly_chart(fig_left, use_container_width=True)
            else:
                st.warning("Brak danych lub niewystarczająca liczba obserwacji do wyznaczenia gęstości.")

        # Po transformacji (przetworzone)
        with col_right:
            st.write("**Po transformacji (przetworzone):**")
            if len(proc_data) >= 2:
                kde_proc = gaussian_kde(proc_data)
                kde_proc.covariance_factor = lambda: kde_proc.factor * bw_adjust
                kde_proc._compute_covariance()
                x_min, x_max = proc_data.min(), proc_data.max()
                margin = (x_max - x_min) * 0.15 if x_max != x_min else 1.0
                x_grid = np.linspace(x_min - margin, x_max + margin, 300)
                y_grid = kde_proc(x_grid)

                fig_right = go.Figure()
                fig_right.add_trace(go.Scatter(
                    x=x_grid,
                    y=y_grid,
                    mode="lines",
                    name="Przetworzone",
                    line=dict(width=2.5, color="rgb(255, 127, 14)"),
                    fill="tozeroy",
                    fillcolor="rgba(255, 127, 14, 0.2)"
                ))
                fig_right.update_layout(
                    xaxis_title=selected_transform,
                    yaxis_title="Gęstość",
                    showlegend=False
                )
                st.plotly_chart(fig_right, use_container_width=True)
            else:
                st.warning("Brak danych lub niewystarczająca liczba obserwacji do wyznaczenia gęstości.")

    else:
        col_left, col_right = st.columns(2)

        # Przed transformacją (surowe)
        with col_left:
            st.write("**Przed transformacją (surowe):**")
            raw_series = df_raw_col[raw_col].fillna("missing").astype(str)
            cat_raw = raw_series.value_counts(normalize=True).reset_index()
            cat_raw.columns = [raw_col, "Share"]
            cat_raw = cat_raw.sort_values(by=raw_col)

            fig_left = px.bar(
                cat_raw,
                x=raw_col,
                y="Share",
                text=cat_raw["Share"].apply(lambda x: f"{x*100:.1f}%"),
                color_discrete_sequence=["rgb(31, 119, 180)"]
            )
            fig_left.update_layout(
                yaxis_title="Udział",
                xaxis=dict(type='category')
            )
            st.plotly_chart(fig_left, use_container_width=True)

        # Po transformacji (przetworzone)
        with col_right:
            st.write("**Po transformacji (przetworzone):**")
            proc_series = df[selected_transform].fillna("missing").astype(str)
            cat_proc = proc_series.value_counts(normalize=True).reset_index()
            cat_proc.columns = [selected_transform, "Share"]
            cat_proc = cat_proc.sort_values(by=selected_transform)

            fig_right = px.bar(
                cat_proc,
                x=selected_transform,
                y="Share",
                text=cat_proc["Share"].apply(lambda x: f"{x*100:.1f}%"),
                color_discrete_sequence=["rgb(255, 127, 14)"]
            )
            fig_right.update_layout(
                yaxis_title="Udział",
                xaxis=dict(type='category')
            )
            st.plotly_chart(fig_right, use_container_width=True)


Overwriting pages/2_Exploratory_Data_Analysis.py


In [16]:
%%file pages/3_Client_Explorer.py

import streamlit as st
import pandas as pd
import joblib
import numpy as np

st.title("👤 Eksplorator klientów")

# --------------------
# Dane
# --------------------

model = joblib.load("best_credit_model.pkl")
df = pd.read_csv("data/app_valid_processed.csv")

X = df.drop(columns=["TARGET"])
y = df["TARGET"]

# --------------------
# Wybór klienta
# --------------------

client_id = st.number_input(
    "Wybierz klienta (indeks)",
    min_value=0,
    max_value=len(X)-1,
    value=0,
    step=1
)

client = X.iloc[[client_id]].copy()

# --------------------
# Predykcja
# --------------------

probability = model.predict_proba(client)[0,1]

# --------------------
# Modyfikacja cech klienta (What-If) - 3 pola w jednym wierszu
# --------------------

st.subheader("Modyfikacja cech klienta (What-If)")
col_in1, col_in2, col_in3 = st.columns(3)

with col_in1:
    new_income = st.slider(
        "Mnożnik dochodu",
        min_value=0.5,
        max_value=3.0,
        value=1.0,
        step=0.1
    )

with col_in2:
    new_credit = st.slider(
        "Mnożnik wysokości kredytu",
        min_value=0.5,
        max_value=2.0,
        value=1.0,
        step=0.1
    )

with col_in3:
    age_shift = st.slider(
        "Zmiana wieku (w latach)",
        min_value=-20,
        max_value=20,
        value=0
    )

modified_client = client.copy()

# Słownik z parametrami transformacji (IQR clipping + Z-score) wyliczony z danych treningowych
@st.cache_data
def get_dynamic_stats():
    from sklearn.model_selection import train_test_split
    df_raw = pd.read_csv("data/application_train_raw_subset.csv.gz")
    X_raw = df_raw.drop(columns=['TARGET'])
    y_raw = df_raw['TARGET']
    X_train_raw, _, _, _ = train_test_split(X_raw, y_raw, test_size=0.3, stratify=y_raw, random_state=42)

    df_train_raw = X_train_raw.copy()
    df_train_raw["LOAN_TO_GOOD"] = df_train_raw["AMT_CREDIT"] / df_train_raw["AMT_GOODS_PRICE"]
    df_train_raw["CREDIT_DURATION"] = df_train_raw["AMT_CREDIT"] / df_train_raw["AMT_ANNUITY"]
    df_train_raw["ANNUITY_TO_INCOME"] = df_train_raw["AMT_ANNUITY"] / df_train_raw["AMT_INCOME_TOTAL"]

    target_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "LOAN_TO_GOOD", "CREDIT_DURATION", "ANNUITY_TO_INCOME"]
    dynamic_stats = {}
    for col in target_cols:
        series = df_train_raw[col].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        
        series_clipped = np.clip(series, lower, upper)
        mean = series_clipped.mean()
        std = series_clipped.std()
        
        dynamic_stats[col] = {
            "mean": mean,
            "std": std,
            "lower": lower,
            "upper": upper
        }
    return dynamic_stats

STATS = get_dynamic_stats()

# Odskalowanie do wartości surowych (odpowiadających skali po obcięciu outlierów)
raw_income_orig = client["AMT_INCOME_TOTAL"].values[0] * STATS["AMT_INCOME_TOTAL"]["std"] + STATS["AMT_INCOME_TOTAL"]["mean"]
raw_credit_orig = client["AMT_CREDIT"].values[0] * STATS["AMT_CREDIT"]["std"] + STATS["AMT_CREDIT"]["mean"]
raw_annuity_orig = client["AMT_ANNUITY"].values[0] * STATS["AMT_ANNUITY"]["std"] + STATS["AMT_ANNUITY"]["mean"]
raw_loan_to_good_orig = client["LOAN_TO_GOOD"].values[0] * STATS["LOAN_TO_GOOD"]["std"] + STATS["LOAN_TO_GOOD"]["mean"]

# Zastosowanie zmian wprowadzonych suwakami
new_raw_income = raw_income_orig * new_income
new_raw_credit = raw_credit_orig * new_credit

# Wyliczenie nowych cech inżynieryjnych w skali surowej
new_loan_to_good_val = new_credit * raw_loan_to_good_orig
new_credit_duration_val = new_raw_credit / raw_annuity_orig
new_annuity_to_income_val = raw_annuity_orig / new_raw_income

# Pomocnicza funkcja do obcięcia wartości odstających i wyliczenia Z-score
def clip_and_scale(val, name):
    clipped = np.clip(val, STATS[name]["lower"], STATS[name]["upper"])
    return (clipped - STATS[name]["mean"]) / STATS[name]["std"]

# Zapisanie przeskalowanych wartości do rekordu klienta
modified_client["AMT_INCOME_TOTAL"] = clip_and_scale(new_raw_income, "AMT_INCOME_TOTAL")
modified_client["AMT_CREDIT"] = clip_and_scale(new_raw_credit, "AMT_CREDIT")
modified_client["DAYS_BIRTH"] = modified_client["DAYS_BIRTH"] - age_shift*365
modified_client["LOAN_TO_GOOD"] = clip_and_scale(new_loan_to_good_val, "LOAN_TO_GOOD")
modified_client["CREDIT_DURATION"] = clip_and_scale(new_credit_duration_val, "CREDIT_DURATION")
modified_client["ANNUITY_TO_INCOME"] = clip_and_scale(new_annuity_to_income_val, "ANNUITY_TO_INCOME")

new_probability = model.predict_proba(modified_client)[0,1]

# --------------------
# Wyniki analizy - 3 pola w jednym wierszu (bez powtórzonego prawdopodobieństwa)
# --------------------

st.subheader("Wyniki analizy ryzyka")
col_out1, col_out2, col_out3 = st.columns(3)

with col_out1:
    st.metric(
        "Oryginalne PD",
        f"{100*probability:.2f}%"
    )

with col_out2:
    st.metric(
        "Zmodyfikowane PD",
        f"{100*new_probability:.2f}%",
        delta=f"{100*(new_probability-probability):.2f}%"
    )

real_target = y.iloc[client_id]
target_text = "TAK" if int(real_target) == 1 else "NIE"

with col_out3:
    st.metric(
        "Czy klient zdefaultował?",
        target_text
    )

# --------------------
# Ocena ryzyka (na podstawie zmodyfikowanego PD)
# --------------------

if new_probability < 0.20:
    st.success("🟢 NISKIE RYZYKO")

elif new_probability < 0.50:
    st.warning("🟡 ŚREDNIE RYZYKO")

else:
    st.error("🔴 WYSOKIE RYZYKO")

# --------------------
# Dane klienta
# --------------------

st.subheader("Cechy klienta")

client_display = client.T.reset_index()
client_display.columns = ["Nazwa zmiennej", "Wartość zmiennej"]

def format_val(val):
    if pd.isna(val):
        return ""
    try:
        num_val = float(val)
        if num_val.is_integer():
            return str(int(num_val))
        return f"{num_val:.2f}"
    except (ValueError, TypeError):
        return str(val)

client_display["Wartość zmiennej"] = client_display["Wartość zmiennej"].apply(format_val)

st.dataframe(client_display, use_container_width=True, hide_index=True)


Overwriting pages/3_Client_Explorer.py


In [17]:
%%file pages/4_Credit_Scoring.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.title("💳 Symulator scoringu kredytowego")

# --------------------
# Wczytanie modelu i danych
# --------------------

model = joblib.load("best_credit_model.pkl")
train = pd.read_csv("data/app_train_processed.csv")
X_train = train.drop(columns=["TARGET"])

# --------------------
# Formularz klienta
# --------------------

st.subheader("Dane klienta")

amt_income = st.number_input(
    "Roczny dochód (AMT_INCOME_TOTAL)",
    min_value=1,
    value=150000
)

amt_credit = st.number_input(
    "Kwota kredytu (AMT_CREDIT)",
    min_value=1,
    value=300000
)

amt_annuity = st.number_input(
    "Rata / annuity (AMT_ANNUITY)",
    min_value=1,
    value=25000
)

# Bezpośrednia kontrola wskaźnika Kredyt/Cena (LOAN_TO_GOOD) zamiast ceny towaru
loan_to_good_val = st.slider(
    "Wskaźnik Kredyt/Cena (LOAN_TO_GOOD)",
    min_value=0.1,
    max_value=3.0,
    value=1.0,
    step=0.05,
)

age_years = st.slider(
    "Wiek (w latach)",
    min_value=18,
    max_value=80,
    value=40
)

children = st.selectbox(
    "Liczba dzieci",
    sorted(X_train["CNT_CHILDREN_grouped"].dropna().unique())
)

own_car_map = {"NIE": 0, "TAK": 1}
own_car_choice = st.selectbox(
    "Posiada samochód?",
    options=list(own_car_map.keys())
)
own_car = own_car_map[own_car_choice]

own_realty_map = {"NIE": 0, "TAK": 1}
own_realty_choice = st.selectbox(
    "Posiada nieruchomość?",
    options=list(own_realty_map.keys())
)
own_realty = own_realty_map[own_realty_choice]

# --------------------
# Budowa rekordu i skalowanie Z-score (po IQR clippingu)
# --------------------

# Bierzemy mediany (które są już przeskalowane) z danych treningowych dla pozostałych zmiennych
new_client = X_train.copy().median(numeric_only=True).to_frame().T

# Dla zmiennych tekstowych podstawiamy najczęstszą wartość (mode)
for col in X_train.select_dtypes(include="object").columns:
    new_client[col] = X_train[col].mode()[0]

# Parametry transformacji (IQR clipping + Z-score) wyliczone z df_train6
@st.cache_data
def get_dynamic_stats():
    from sklearn.model_selection import train_test_split
    df_raw = pd.read_csv("data/application_train_raw_subset.csv.gz")
    X_raw = df_raw.drop(columns=['TARGET'])
    y_raw = df_raw['TARGET']
    X_train_raw, _, _, _ = train_test_split(X_raw, y_raw, test_size=0.3, stratify=y_raw, random_state=42)

    df_train_raw = X_train_raw.copy()
    df_train_raw["LOAN_TO_GOOD"] = df_train_raw["AMT_CREDIT"] / df_train_raw["AMT_GOODS_PRICE"]
    df_train_raw["CREDIT_DURATION"] = df_train_raw["AMT_CREDIT"] / df_train_raw["AMT_ANNUITY"]
    df_train_raw["ANNUITY_TO_INCOME"] = df_train_raw["AMT_ANNUITY"] / df_train_raw["AMT_INCOME_TOTAL"]

    target_cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "LOAN_TO_GOOD", "CREDIT_DURATION", "ANNUITY_TO_INCOME"]
    dynamic_stats = {}
    for col in target_cols:
        series = df_train_raw[col].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        
        series_clipped = np.clip(series, lower, upper)
        mean = series_clipped.mean()
        std = series_clipped.std()
        
        dynamic_stats[col] = {
            "mean": mean,
            "std": std,
            "lower": lower,
            "upper": upper
        }
    return dynamic_stats

STATS = get_dynamic_stats()

# Pomocnicza funkcja do obcięcia wartości odstających i wyliczenia Z-score
def clip_and_scale(val, name):
    clipped = np.clip(val, STATS[name]["lower"], STATS[name]["upper"])
    return (clipped - STATS[name]["mean"]) / STATS[name]["std"]

# Podmiana wartości z formularza po odpowiednim przeskalowaniu Z-score i IQR clippingu
new_client["AMT_INCOME_TOTAL"] = clip_and_scale(amt_income, "AMT_INCOME_TOTAL")
new_client["AMT_CREDIT"] = clip_and_scale(amt_credit, "AMT_CREDIT")
new_client["AMT_ANNUITY"] = clip_and_scale(amt_annuity, "AMT_ANNUITY")

# DAYS_BIRTH jest w dniach surowych
new_client["DAYS_BIRTH"] = -age_years * 365

# Dynamiczne wyliczenie, obcięcie i przeskalowanie cech inżynieryjnych
credit_duration_val = amt_credit / amt_annuity
annuity_to_income_val = amt_annuity / amt_income

new_client["LOAN_TO_GOOD"] = clip_and_scale(loan_to_good_val, "LOAN_TO_GOOD")
new_client["CREDIT_DURATION"] = clip_and_scale(credit_duration_val, "CREDIT_DURATION")
new_client["ANNUITY_TO_INCOME"] = clip_and_scale(annuity_to_income_val, "ANNUITY_TO_INCOME")

# Podmiana cech kategorycznych / binarnych z formularza
new_client["CNT_CHILDREN_grouped"] = children
new_client["FLAG_OWN_CAR"] = own_car
new_client["FLAG_OWN_REALTY"] = own_realty

# --------------------
# Predykcja
# --------------------

if st.button("Oblicz ryzyko"):

    proba = model.predict_proba(new_client)
    probability = proba[0,1]

    st.metric(
        "Prawdopodobieństwo defaultu (PD)",
        f"{100*probability:.2f}%"
    )

    if probability < 0.20:
        st.success("🟢 NISKIE RYZYKO")

    elif probability < 0.50:
        st.warning("🟡 ŚREDNIE RYZYKO")

    else:
        st.error("🔴 WYSOKIE RYZYKO")

    # Dane klienta (formatowanie tabelaryczne)
    st.subheader("Wprowadzone dane i cechy klienta")
    client_display = new_client.T.reset_index()
    client_display.columns = ["Nazwa zmiennej", "Wartość zmiennej"]

    def format_val(val):
        if pd.isna(val):
            return ""
        try:
            num_val = float(val)
            if num_val.is_integer():
                return str(int(num_val))
            return f"{num_val:.2f}"
        except (ValueError, TypeError):
            return str(val)

    client_display["Wartość zmiennej"] = client_display["Wartość zmiennej"].apply(format_val)
    st.dataframe(client_display, use_container_width=True, hide_index=True)


Overwriting pages/4_Credit_Scoring.py


In [18]:
%%file requirements.txt

streamlit
pandas
numpy
joblib
scikit-learn
scipy
plotly


Overwriting requirements.txt
